In [1]:
from transformers import AutoModelForSeq2SeqLM
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Load mT5-small model
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

# Enable gradient checkpointing right away (saves memory)
model.gradient_checkpointing_enable()
print("mT5 model loaded.")

# 2. Load your three CSV files (adjust paths if they're in a new dataset folder)
small  = pd.read_csv("/kaggle/input/datasets/siddhanttimalsina/mt5-dataset/small.csv")
medium = pd.read_csv("/kaggle/input/datasets/siddhanttimalsina/mt5-dataset/medium.csv")
large  = pd.read_csv("/kaggle/input/datasets/siddhanttimalsina/mt5-dataset/large.csv")

# 3. Combine
all_data = pd.concat([small, medium, large], ignore_index=True)
print("Total rows:", all_data.shape[0])

# 4. Train / val / test split (80/10/10)
train_df, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", train_df.shape[0], " | Val:", val_df.shape[0], " | Test:", test_df.shape[0])

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

mT5 model loaded.
Total rows: 50529
Train: 40423  | Val: 5053  | Test: 5053


In [2]:
from datasets import Dataset

# Convert DataFrames to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df[['content', 'summary']])
val_dataset   = Dataset.from_pandas(val_df[['content', 'summary']])
test_dataset  = Dataset.from_pandas(test_df[['content', 'summary']])

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def tokenize_function(examples):
    # Tokenize source articles
    model_inputs = tokenizer(
        examples['content'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    # Tokenize target summaries – T5 uses text_target
    labels = tokenizer(
        text_target=examples['summary'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("Tokenizing training set...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing validation set...")
tokenized_val   = val_dataset.map(tokenize_function, batched=True)
print("Tokenizing test set...")
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

# Remove raw text columns
tokenized_train = tokenized_train.remove_columns(['content', 'summary'])
tokenized_val   = tokenized_val.remove_columns(['content', 'summary'])
tokenized_test  = tokenized_test.remove_columns(['content', 'summary'])

print("Tokenization complete.")
print("Train features:", tokenized_train.features)

Tokenizing training set...


Map:   0%|          | 0/40423 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [ ]:
import torch
torch.cuda.empty_cache()

from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-nepali-summarizer",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=500,
    per_device_train_batch_size=2,            # per GPU
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,            # effective batch = 2 * 2 GPUs * 8 = 32
    num_train_epochs=3,
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

print("Training arguments ready.")
print("Effective batch size:", training_args.per_device_train_batch_size * 2 * training_args.gradient_accumulation_steps)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Starting training on 2 GPUs...")
trainer.train()
print("Training finished.")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/working"):
    for name in dirs + files:
        path = os.path.join(root, name)
        try:
            size = os.path.getsize(path)
        except:
            size = 0
        if size > 100_000_000:   # files larger than 100 MB
            print(f"{size//1024//1024:>5} MB  {path}")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("./mt5-nepali-summarizer/checkpoint-3792")
print("Best checkpoint loaded into memory.")

In [ ]:
import shutil
shutil.rmtree("./mt5-nepali-summarizer", ignore_errors=True)
print("Training folder deleted.")

In [ ]:
shutil.rmtree("./mt5-final", ignore_errors=True)

In [ ]:
model.save_pretrained("./mt5-final")
tokenizer.save_pretrained("./mt5-final")
print("Final model saved to ./mt5-final")

In [ ]:
shutil.make_archive("/kaggle/working/mt5-final", 'zip', "./mt5-final")
print("ZIP created: /kaggle/working/mt5-final.zip")